# Unit 3 — ensembles, Naive Bayes, HMM, GMM

Phase 7. Ensembles and feature importance, the Naive Bayes prior that routes the pipeline, the HMM that labels node roles (S4), and the GMM shape vocabulary.

This notebook reads committed results from `reports/`. It reproduces nothing and needs no corpus or GPU. `reports/README.md` names the command that regenerates each file. The full syllabus mapping is in `docs/syllabus_map.md`.

In [1]:
import json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "reports").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CELLS = json.loads((ROOT / "reports" / "master_results.json").read_text(encoding="utf-8"))["cells"]


def results(*stages):
    """The master table's rows for these stages, as a Markdown table."""
    rows = ["| stage | model | metric | value | target | verdict |", "| :-- | :-- | :-- | --: | --: | :-: |"]
    for c in CELLS:
        if c["stage"] in stages and c.get("available", True):
            verdict = "" if c.get("passes") is None else ("pass" if c["passes"] else "**fail**")
            target = "" if c.get("target") is None else c["target"]
            rows.append(f"| {c['stage']} | {c['model']} | {c['metric']} | {c['value']} | {target} | {verdict} |")
    display(Markdown("\n".join(rows)))


def report(name, *keys):
    """Selected top-level fields of one reports/*.json."""
    data = json.loads((ROOT / "reports" / name).read_text(encoding="utf-8"))
    return {k: data[k] for k in keys if k in data}


## Feature importance (7.1)

Gini and SHAP give the same ranking (Spearman 0.984). Permutation importance, the one measured on held-out rows, disagrees with both.

![p7_shap_summary.png](../reports/figures/p7_shap_summary.png)

`reports/figures/p7_shap_summary.png`

![p7_occlusion.png](../reports/figures/p7_occlusion.png)

`reports/figures/p7_occlusion.png`

## HMM + Viterbi — S4 (7.3)

The margin over the target is 0.0003 and the seed spread is 0.0010. The target is met, not beaten.

In [2]:
results("parse")
report("s4_role_labelling.json", "sequences", "nodes", "macro_f1", "macro_f1_std", "seeds_clearing_target", "seeds")

| stage | model | metric | value | target | verdict |
| :-- | :-- | :-- | --: | --: | :-: |
| parse | HMM + per-component Viterbi | macro F1 | 0.8003 | 0.8 | pass |
| parse | HMM + per-component Viterbi | macro F1 std | 0.001 |  |  |
| parse | HMM + per-component Viterbi | accuracy | 0.7873 |  |  |

{'sequences': 993,
 'nodes': 14056,
 'macro_f1': 0.8003,
 'macro_f1_std': 0.001,
 'seeds_clearing_target': 6,
 'seeds': 10}

![p7_hmm_transitions.png](../reports/figures/p7_hmm_transitions.png)

`reports/figures/p7_hmm_transitions.png`

![p7_hmm_emissions.png](../reports/figures/p7_hmm_emissions.png)

`reports/figures/p7_hmm_emissions.png`

![p7_baumwelch.png](../reports/figures/p7_baumwelch.png)

`reports/figures/p7_baumwelch.png`

## GMM + EM (7.4)

![p7_gmm_em.png](../reports/figures/p7_gmm_em.png)

`reports/figures/p7_gmm_em.png`

![p7_gmm_selection.png](../reports/figures/p7_gmm_selection.png)

`reports/figures/p7_gmm_selection.png`

![p7_gmm_montage.png](../reports/figures/p7_gmm_montage.png)

`reports/figures/p7_gmm_montage.png`

![p7_scribe_covariance.png](../reports/figures/p7_scribe_covariance.png)

`reports/figures/p7_scribe_covariance.png`